# Lecture 1.4: External Models with Custom Provider


## Retrieve OpenAI API Key from Secrets

Your OpenAI API key is stored in Databricks Secrets:
- **Scope**: `llmops_course`
- **Key**: `openai_key`

### How to Access Secrets:

**Using Databricks SDK** (recommended - works everywhere):
```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
openai_api_key = w.secrets.get_secret(scope="llmops_course", key="openai_key").value
```

**Using dbutils** (Databricks notebooks only):
```python
openai_api_key = dbutils.secrets.get(scope="llmops_course", key="openai_key")
```

**For External Model Endpoints** (UI configuration):
- Use secret reference format: `{{secrets/llmops_course/openai_key}}`


## Create External Model Endpoint with Custom Provider


In [ ]:
import mlflow.deployments
from loguru import logger

# Get MLflow Deployments client
client = mlflow.deployments.get_deploy_client("databricks")

# Configuration
ENDPOINT_NAME = "openai-dalle-custom"

# Check if endpoint already exists
try:
    existing = client.get_endpoint(ENDPOINT_NAME)
    logger.info(f"Endpoint '{ENDPOINT_NAME}' already exists")
    logger.info(f"Status: {existing}")
except Exception:
    logger.info(f"Creating External Model endpoint: {ENDPOINT_NAME}")

    # Create External Model endpoint for OpenAI DALL-E
    endpoint = client.create_endpoint(
        name=ENDPOINT_NAME,
        config={
            "served_entities": [{
                "name": "dalle-image-generation",
                "external_model": {
                    "name": "dall-e-3",
                    "provider": "openai",
                    "task": "llm/v1/images",  # Image generation task type
                    "openai_config": {
                        "openai_api_key": "{{secrets/llmops_course/openai_key}}",
                        "openai_api_base": "https://api.openai.com/v1",
                        "openai_api_type": "openai"
                    }
                }
            }]
        }
    )

    logger.info(f"Endpoint created successfully: {ENDPOINT_NAME}")
    logger.info(f"Configuration: {endpoint}")


## 6. Query the Custom Provider Endpoint

Once your endpoint is deployed, you can query it using the OpenAI SDK:


In [ ]:
from databricks.sdk import WorkspaceClient
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
import json

w = WorkspaceClient()

# Authenticate using Databricks SDK
host = w.config.host
token = w.tokens.create(lifetime_seconds=1200).token_value

# Create OpenAI client pointing to Databricks endpoint
client = OpenAI(
    api_key=token,
    base_url=f"{host.rstrip('/')}/serving-endpoints"
)

ENDPOINT_NAME = "openai-dalle-custom"

logger.info(f"Client configured to use endpoint: {ENDPOINT_NAME}")
logger.info(f"Base URL: {host}/serving-endpoints")


## Generate an Image (Base64 Response)

### Response Format Options:
- **`b64_json`**: Returns image as base64-encoded string (recommended)


In [ ]:
# Generate image with base64 response
response = client.images.generate(
    model=ENDPOINT_NAME,
    prompt="Two cats wearing superhero capes in a sunny garden",
    n=1,  # Number of images to generate
    style="vivid",  # Options: "vivid" or "natural"
    quality="standard",  # Options: "standard" or "hd"
    response_format="b64_json"  # Returns base64-encoded image
)

logger.info("Image generated successfully!")
logger.info(f"Prompt: {response.data[0].revised_prompt if hasattr(response.data[0], 'revised_prompt') else 'N/A'}")
logger.info("Response format: b64_json")


## 8. Display the Generated Image


In [ ]:
# Decode and display the image
image_data = response.data[0].b64_json
image_bytes = base64.b64decode(image_data)
image = Image.open(BytesIO(image_bytes))

# Display in notebook
display(image)

# Optionally save to file
# image.save("/dbfs/tmp/generated_image.png")
logger.info(f"Image size: {image.size}")
logger.info(f"Image format: {image.format}")


## Generate Image with URL Response


In [ ]:
# Generate image with URL response (temporary access)
response_url = client.images.generate(
    model=ENDPOINT_NAME,
    prompt="A futuristic data center with glowing servers",
    n=1,
    style="vivid",
    quality="standard",
    response_format="url"  # Returns temporary URL
)

image_url = response_url.data[0].url
logger.info("Image generated!")
logger.info("Temporary URL (expires in 2 hours):")
logger.info(image_url)
